In [ ]:
from langchain_deepseek import ChatDeepSeek
from typing import TypedDict
from langgraph.constants import START,END
from langgraph.graph import StateGraph

from dotenv import load_dotenv

# 加载.env文件中的配置
load_dotenv(override=True)

# 定义模型
model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 定义状态
class OverAllstate(TypedDict):
    topic: str
    poem: str
    joke: str

# 定义节点
def node_a(state:OverAllstate)->OverAllstate:
    poem =model.invoke([f"写一首关于{state['topic']}主题的诗"]).content
    return {
        "poem":poem
    }


# 定义节点
def node_b(state:OverAllstate)->OverAllstate:
    joke =model.invoke([f"写一个关于{state['topic']}主题的笑话"]).content
    return {
        "joke":joke
    }


# 构建图

builder= StateGraph(state_schema=OverAllstate)
# 添加节点
builder.add_node("node_a",node_a);
builder.add_node("node_b",node_b);
# 添加边
builder.add_edge(START,"node_a");
builder.add_edge(START,"node_b");
builder.add_edge("node_a",END);
builder.add_edge("node_b",END);

# 获取图
graph = builder.compile()
# 运行图
result=graph.invoke({"topic":"小猫"})
print(result)

from IPython.display import display
display(graph)
